# Pipeline NLP Video Pemrograman V2

Colab melatih dan menguji model pada dataset bersumber yang punya acuan. Streamlit menyediakan demo video pemrograman dengan narasi sintetis. Angka benchmark di notebook mengukur transkripsi dan peringkasan secara terpisah.

## 01. Setup Colab

Pilih runtime GPU, jalankan sel ini, lalu unggah **paket-pemrograman-v2.zip** saat diminta. Jika paket sudah tersedia di panel Files, sel langsung memakainya.

In [ ]:
# Lokasi paket proyek di Colab
ZIP_PROYEK = "/content/paket-pemrograman-v2.zip"

import json
import os
import shutil
import subprocess
import sys
from datetime import datetime
from io import BytesIO
from pathlib import Path
from zipfile import BadZipFile, ZipFile

from IPython.display import Markdown, display

try:
    from google import colab
except ImportError:
    colab = None
IS_COLAB = colab is not None
if IS_COLAB:
    ROOT_COLAB = Path("/content/pemrograman-v2")
    required_files = {
        "requirements-training.txt", "requirements-notebooks.txt",
        "notebooks/Pipeline_NLP_Video_Pemrograman_V2.ipynb",
        "scripts/notebook_training.py", "scripts/compare_partial_seq2seq.py",
        "data/processed/indosum/train.csv", "data/processed/multilingual_code/test.csv",
        "data/processed/nss-id/test.csv", "data/processed/lumina_split/test.csv",
    }
    bundle_version = "pemrograman-v2"
    version_file = "colab-bundle-version.txt"

    def bundle_problem(package):
        if version_file not in package.namelist():
            return "ZIP ini berasal dari paket lama dan belum memiliki penanda versi."
        version = package.read(version_file).decode("utf-8", errors="replace").strip()
        if version != bundle_version:
            return f"Versi paket {version!r}, sedangkan notebook memerlukan {bundle_version!r}."
        missing = sorted(required_files.difference(package.namelist()))
        if missing:
            return f"File wajib belum tersedia di ZIP: {', '.join(missing)}."
        return None

    marker = ROOT_COLAB / version_file
    if (not marker.is_file() or marker.read_text(encoding="utf-8").strip() != bundle_version
            or not all((ROOT_COLAB / name).is_file() for name in required_files)):
        archive = Path(ZIP_PROYEK).expanduser()
        extracted = False
        if archive.is_file():
            try:
                with ZipFile(archive) as package:
                    problem = bundle_problem(package)
                    if problem:
                        print(problem)
                    else:
                        package.extractall(ROOT_COLAB)
                        extracted = True
            except BadZipFile:
                print(f"File {archive.name} bukan ZIP yang utuh.")
        if not extracted:
            from google.colab import files

            print(f"Pilih {archive.name} dari komputer (sekitar 46 MB).")
            uploaded = files.upload()
            zip_files = [(name, content) for name, content in uploaded.items() if Path(name).suffix.lower() == ".zip"]
            if len(zip_files) != 1:
                raise ValueError(f"Pilih satu file {archive.name}, lalu jalankan kembali sel 01.")
            name, content = zip_files[0]
            try:
                with ZipFile(BytesIO(content)) as package:
                    if problem := bundle_problem(package):
                        raise ValueError(f"{name}: {problem} Gunakan {archive.name}.")
                    package.extractall(ROOT_COLAB)
            except BadZipFile as exc:
                raise ValueError(f"{name} bukan ZIP yang utuh. Unduh ulang {archive.name}.") from exc
            del uploaded, zip_files, content
    os.chdir(ROOT_COLAB)
    installed = ROOT_COLAB / ".colab-dependencies-installed"
    if not installed.exists():
        if not shutil.which("ffmpeg"):
            subprocess.run(["apt-get", "update", "-qq"], check=True)
            subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", "requirements-training.txt", "-r", "requirements-notebooks.txt"], check=True)
        installed.touch()

import matplotlib.pyplot as plt
import pandas as pd
import torch

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "requirements-training.txt").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

DEFAULT_DEVICE = "cuda:0" if IS_COLAB and torch.cuda.is_available() else "cpu"

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

import matplotlib.ticker as ticker
from IPython.display import FileLink

DEVICE = DEFAULT_DEVICE
REPORT_PATH = os.getenv("NLP_V2_REPORTS", "").strip()
RUN = Path(REPORT_PATH).expanduser().resolve() if REPORT_PATH else (
    ROOT / "outputs/notebooks/pemrograman-v2" / datetime.now().strftime("%Y%m%d-%H%M%S-%f")
)
BEFORE = "Sebelum pelatihan tambahan"
AFTER = "Sesudah pelatihan tambahan"

def execute(module, *args, python=sys.executable):
    command = [str(python), "-u", "-m", module, *map(str, args)]
    with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace",
                          env={**os.environ, "PYTHONIOENCODING": "utf-8"}) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
        except KeyboardInterrupt:
            process.terminate()
            raise
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)

def table(frame, percentages=()):
    display(frame.style.hide(axis="index").format(
        {column: "{:.1%}" for column in percentages}, precision=2, na_rep="—", escape="html"
    ).set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "normal", "max-width": "42ch"}))

print(f"Perangkat: {DEVICE}")
if IS_COLAB and DEVICE == "cpu":
    print("Untuk latihan, pilih Runtime > Change runtime type > GPU.")


## 02. Dataset dan pembagian

- **NSS-ID Imperative** dan **LUMINA v4** dipakai untuk transkripsi bahasa Indonesia. Manifest NSS-ID memakai rekaman penutur manusia. LUMINA berisi klip penutur asli dengan audio resmi yang dipasangkan ke videonya. [NSS-ID](https://huggingface.co/datasets/Atika88/Indonesian-ASR-11-Class-Dataset) · [LUMINA](https://data.mendeley.com/datasets/8fw93k4rny/4)
- **IndoSum** menguji peringkasan artikel bahasa Indonesia dengan ringkasan acuan. [IndoSum](https://github.com/kata-ai/indosum)
- **The Vault Function** menguji peringkasan fungsi dengan docstring Inggris sebagai acuan. Sampel mencakup sepuluh bahasa pemrograman, termasuk Python. Setiap contoh menyimpan repositori, jalur, commit, dan lisensi asal. [The Vault Function](https://huggingface.co/datasets/Fsoft-AIC/the-vault-function)

Tabel menunjukkan isi paket. Bagian latihan mencatat jumlah contoh yang benar-benar dipakai dan menjaga kelompok penutur atau repositori terpisah antarsplit.

In [ ]:
BASE = ROOT / "data/processed"
DATASETS = [
    ("NSS-ID Imperative", "Transkripsi audio", "Indonesia", "nss-id", "val", "speaker_id"),
    ("LUMINA v4", "Transkripsi audio dari video", "Indonesia", "lumina_split", "validation", "speaker_id"),
    ("IndoSum", "Ringkasan artikel", "Indonesia", "indosum", "validation", "id"),
    ("The Vault Function", "Ringkasan fungsi", "Inggris", "multilingual_code", "validation", "repo"),
]
dataset_counts = []
for name, task, language, folder, validation, group in DATASETS:
    counts = []
    used_groups = set()
    for filename in ("train", validation, "test"):
        frame = pd.read_csv(BASE / folder / f"{filename}.csv", dtype=str, keep_default_na=False)
        groups = set(frame[group])
        if "" in groups or groups & used_groups:
            raise ValueError(f"Grup kosong atau muncul di lebih dari satu split {name}: {filename}")
        used_groups.update(groups)
        counts.append(len(frame))
    dataset_counts.append([name, task, language, *counts])

table(pd.DataFrame(dataset_counts, columns=["Dataset", "Tugas", "Bahasa acuan", "Latih", "Validasi", "Uji"]))

## 03. Pelatihan dan evaluasi transkripsi

Whisper membaca audio karena model peringkas menerima teks. Setiap korpus punya skor WER dan CER sendiri. WER menghitung kesalahan kata dan CER menghitung kesalahan karakter. Skor lebih rendah lebih baik.

Enam epoch memberi titik awal untuk melihat perubahan skor validasi. Ubah jumlah epoch jika perlu. Patience nol menjalankan semua epoch. Pilih nilai dua jika ingin berhenti setelah dua epoch tanpa perbaikan. Checkpoint dipilih memakai rata-rata WER validasi, lalu CER sebagai pemecah seri.

In [ ]:
LATIH_WHISPER = True
EPOCH_WHISPER = 6
AKUMULASI_WHISPER = 1
PATIENCE_WHISPER = 0

whisper_dir = RUN / "whisper"
whisper_result = whisper_dir / "report.json"
if IS_COLAB and LATIH_WHISPER and not REPORT_PATH:
    execute("scripts.notebook_training", "whisper", "--profile", "pilot", "--device", DEVICE,
            "--output", whisper_dir, "--epochs", EPOCH_WHISPER, "--patience", PATIENCE_WHISPER,
            "--accumulation", AKUMULASI_WHISPER, "--corpus", "NSS-ID", "--corpus", "LUMINA", "--full-splits")
whisper_report = read_json(whisper_result) if whisper_result.is_file() else None
if whisper_report:
    print(f"Epoch dijalankan: {len(whisper_report['history'])} | Epoch terpilih: {whisper_report['best_epoch']}")
    splits = pd.DataFrame.from_dict(whisper_report["splits"], orient="index")
    table(splits.rename_axis("Dataset").reset_index().rename(
        columns={"train": "Latih", "validation": "Validasi", "test": "Uji"}))
else:
    print("Laporan Whisper akan tampil setelah latihan selesai di Colab.")

In [ ]:
if whisper_report:
    asr_rows = []
    for name, pair in whisper_report["test"].items():
        for variant, label in (("base", BEFORE), ("adapted", AFTER)):
            predictions = pd.DataFrame(whisper_report["test_predictions"][name][variant])
            errors = predictions[["Penggantian", "Penghapusan", "Penyisipan"]].sum().to_dict()
            asr_rows.append({"Dataset": name, "Tahap model": label, "Klip uji": len(predictions),
                             "WER": pair[variant]["wer"], "CER": pair[variant]["cer"], **errors})
        with_errors = predictions.sort_values(["wer", "id"], ascending=[False, True]).head(3)
        display(Markdown(f"### Contoh dengan WER tertinggi · {name}"))
        table(with_errors[["id", "reference", "prediction", "wer"]].rename(columns={
            "id": "Klip", "reference": "Acuan", "prediction": "Transkrip", "wer": "WER",
        }), ("WER",))
    display(Markdown("### Hasil uji per dataset"))
    table(pd.DataFrame(asr_rows), ("WER", "CER"))
    history = pd.DataFrame(whisper_report["history"]).set_index("epoch")
    ax = history[["macro_wer", "macro_cer"]].rename(
        columns={"macro_wer": "WER", "macro_cer": "CER"}
    ).plot(marker="o", figsize=(7, 3), xlabel="Epoch", title="Rata-rata kesalahan pada validasi")
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
    plt.tight_layout()
    plt.show()

## 04. Pelatihan peringkas

- **IndoSum** membandingkan mBART, mT5, dan IndoBART v2 pada ringkasan bahasa Indonesia. Setelan awal memakai 500 artikel latih, 50 validasi, dan 100 uji dari split yang sesuai.
- **The Vault Function** membandingkan mBART dan mT5 pada 50/20/20 pasangan kode dan docstring Inggris. IndoBART v2 menggunakan token bahasa Indonesia, Jawa, dan Sunda. Hasil The Vault mengukur ringkasan fungsi serta memakai acuan berbahasa Inggris.
- Kedua benchmark melatih model secara terpisah. mT5 memakai LoRA. mBART dan IndoBART memperbarui proyeksi Q/V pada cross-attention decoder terakhir. Model dan epoch dipilih dengan ROUGE-L validasi.

In [ ]:
from scripts.compare_partial_seq2seq import BENCHMARK_MODELS, comparison_reports

LATIH_PERINGKAS = True
EPOCH_PERINGKAS = 6
AKUMULASI_PERINGKAS = 4
PATIENCE_PERINGKAS = 0
INDOSUM_LATIH, INDOSUM_VALIDASI, INDOSUM_UJI = 500, 50, 100

summary_dir = RUN / "peringkas"
benchmarks = {"indosum": "IndoSum", "multilingual_code": "The Vault Function"}
model_names = {"mbart": "mBART Large", "mt5": "mT5 Small", "indobart-v2": "IndoBART v2"}
if IS_COLAB and LATIH_PERINGKAS and not REPORT_PATH:
    for key in BENCHMARK_MODELS["indosum"]:
        selected = [flag for dataset in benchmarks if key in BENCHMARK_MODELS[dataset]
                    for flag in ("--benchmark", dataset)]
        execute("scripts.compare_partial_seq2seq", "--profile", "pilot", "--model", key,
                "--device", DEVICE, "--download", "--output", summary_dir, "--epochs", EPOCH_PERINGKAS,
                "--accumulation", AKUMULASI_PERINGKAS, "--patience", PATIENCE_PERINGKAS,
                "--indosum-limits", INDOSUM_LATIH, INDOSUM_VALIDASI, INDOSUM_UJI, *selected)
summary_reports = comparison_reports(summary_dir, datasets=benchmarks) if summary_dir.is_dir() else []
if not summary_reports:
    print("Laporan peringkasan akan tampil setelah latihan selesai di Colab.")

## 05. Evaluasi peringkas

Bandingkan sebelum dan sesudah pelatihan pada contoh uji yang sama. ROUGE-1 mengukur tumpang tindih kata, ROUGE-2 pasangan kata, dan ROUGE-L urutan kata. Semua ditampilkan sebagai persentase. Keluaran kosong, panjang input yang terpotong, dan contoh prediksi membantu membaca skor.

In [ ]:
for dataset, name in benchmarks.items():
    group = [row for row in summary_reports if row["dataset"] == dataset]
    if not group:
        continue
    display(Markdown(f"### {name}"))
    scores, examples, language_scores, history_rows = [], [], [], []
    for row in group:
        label = model_names[row["model"]]
        for metric_key, label_stage in (("base", BEFORE), ("test", AFTER)):
            score = row[metric_key]
            scores.append({"Model": label, "Tahap model": label_stage, "Latih": row["counts"]["train"],
                           "Validasi": row["counts"]["validation"], "Uji": row["counts"]["test"],
                           "Epoch dijalankan": len(row["history"]), "Epoch terpilih": row["best_epoch"],
                           "ROUGE-1": score["rouge1"], "ROUGE-2": score["rouge2"],
                           "ROUGE-L": score["rougeL"], "Kosong": score["empty"], "Input terpotong": score["truncated"],
                           "Median inferensi (ms)": score["latency_ms"]})
        for epoch in row["history"]:
            history_rows.append({"Epoch": epoch["epoch"], "Model": label, "ROUGE-L": epoch["validation"]["rougeL"]})
        base_predictions = {item["id"]: item["prediction"] for item in row["base_predictions"]}
        predictions = sorted(row["test_predictions"], key=lambda item: (item["rougeL"], item["id"]))
        for item in predictions[:2]:
            examples.append({"Model": label, "ID": item["id"], "Acuan": item["reference"],
                             "Sebelum": base_predictions[item["id"]], "Sesudah": item["prediction"],
                             "ROUGE-L sesudah": item["rougeL"]})
        if dataset == "multilingual_code":
            per_language = pd.DataFrame(predictions).groupby("language")["rougeL"].agg(["size", "mean"]).reset_index()
            language_scores.extend({"Model": label, "Bahasa pemrograman": entry.language,
                                    "Contoh uji": entry.size, "ROUGE-L": entry.mean}
                                   for entry in per_language.itertuples(index=False))
    table(pd.DataFrame(scores), ("ROUGE-1", "ROUGE-2", "ROUGE-L"))
    curve = pd.DataFrame(history_rows).pivot(index="Epoch", columns="Model", values="ROUGE-L")
    ax = curve.plot(marker="o", figsize=(7, 3), title=f"ROUGE-L validasi · {name}")
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
    plt.tight_layout()
    plt.show()
    if language_scores:
        display(Markdown("**Hasil sesudah fine-tuning per bahasa pemrograman**"))
        table(pd.DataFrame(language_scores), ("ROUGE-L",))
    display(Markdown("**Contoh dengan ROUGE-L terendah per model**"))
    table(pd.DataFrame(examples), ("ROUGE-L sesudah",))

## 06. Laporan dan demo

Laporan JSON menyimpan konfigurasi, epoch terpilih, metrik, dan pasangan prediksi dengan acuannya. Simpan folder hasil ke Drive jika ingin membukanya lagi. Untuk membuka laporan di lingkungan lokal, arahkan variabel lingkungan `NLP_V2_REPORTS` ke folder hasil yang berisi subfolder `whisper` dan `peringkas`.

Di Streamlit, pilih **Jalankan demo pandas** untuk melihat transkripsi, OCR/Gemini ketika tersedia, serta ringkasan dari MP4 bernarasi sintetis.

In [ ]:
if whisper_report or summary_reports:
    print(f"Folder hasil: {RUN}")
    for report in sorted(RUN.rglob("report.json")):
        display(FileLink(str(report)))